In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

We have imported the data and downloaded via the url: https://stooq.com/q/d/?f=20160624&t=20260624&s=^spx&c=0

In [2]:
csv_path = Path.cwd().parent / "data" / "raw" / "^spx_d.csv"

df = pd.read_csv(csv_path, parse_dates=["Date"]).set_index("Date").sort_index()
prices = df["Close"].dropna()

log_returns = np.log(prices / prices.shift(1)).dropna()
returns_pct = 100*log_returns          # percent scale for arch

print(f"{len(returns_pct)} daily returns, "
      f"{prices.index[0].date()} to {prices.index[-1].date()}")
print(f"mean {returns_pct.mean():.4f}%, std {returns_pct.std():.4f}%")

2511 daily returns, 2016-06-24 to 2026-06-23
mean 0.0512%, std 1.1421%


# GARCH(1,1): Setup, Theory, and Estimation from Scratch

We fit GARCH(1,1) with the `arch` library, but this cell documents the full model
and the estimator we *could* write ourselves - the recursion, the likelihood, and
the constrained optimisation that `arch` performs internally.

## Motivation: two notions of volatility

- **Implied vol** - backed out of option *prices* (Week 1 smiles). Forward-looking,
  the market's risk-neutral expectation, varies by strike and maturity.
- **Realised vol** - estimated from the underlying's *historical returns*.
  Backward-looking, physical measure. GARCH lives here.

The empirical fact a constant $\sigma$ ignores: **volatility clustering** - large
moves follow large moves, calm follows calm. Vol is persistent, not constant, and
not independent across days.

## The model

$$r_t = \mu + \epsilon_t, \qquad \epsilon_t = \sigma_t z_t, \quad z_t \sim N(0,1)$$
$$\sigma_t^2 = \omega + \alpha\,\epsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2$$

Today's variance = baseline $\omega$ + reaction to yesterday's shock
($\alpha\epsilon_{t-1}^2$) + persistence of yesterday's variance ($\beta\sigma_{t-1}^2$).
The recursion manufactures clustering: a big shock raises $\sigma_t^2$, which tends to
produce another big shock, decaying over time.

## Structure: GARCH(1,1) is ARMA(1,1) on squared shocks

Define the squared-shock innovation $\nu_t = \epsilon_t^2 - \sigma_t^2$ (mean-zero by
construction, since $\mathbb{E}[\epsilon_t^2\mid\mathcal{F}_{t-1}] = \sigma_t^2$).
Substituting $\sigma_t^2 = \epsilon_t^2 - \nu_t$ into the recursion:

$$\epsilon_t^2 = \omega + (\alpha+\beta)\,\epsilon_{t-1}^2 + \nu_t - \beta\,\nu_{t-1}$$

This is **ARMA(1,1) in $\epsilon_t^2$**: AR coefficient $(\alpha+\beta)$, MA coefficient
$-\beta$, intercept $\omega$. Returns are white noise in *level* (unpredictable); their
*squares* carry the ARMA structure (volatility *is* predictable).

Consequences inherited directly from AR(1) theory:

| Quantity | Formula | Meaning |
|---|---|---|
| Persistence | $\alpha + \beta$ | AR(1) coefficient on $\epsilon_t^2$ |
| Stationarity | $\alpha + \beta < 1$ | finite long-run variance (the $|\phi|<1$ condition) |
| Long-run variance | $\bar\sigma^2 = \dfrac{\omega}{1-\alpha-\beta}$ | level vol mean-reverts to |
| Mean-reversion | $\mathbb{E}_t[\sigma_{t+k}^2] - \bar\sigma^2 = (\alpha+\beta)^k(\sigma_t^2 - \bar\sigma^2)$ | geometric decay |
| Half-life | $\dfrac{\ln 0.5}{\ln(\alpha+\beta)}$ days | time for a vol spike to fade halfway |

$\alpha+\beta = 1$ is the boundary (IGARCH) - shocks to variance never decay.
For daily equities $\alpha+\beta \approx 0.95$-$0.99$, half-life ~weeks to a quarter.

## Estimation: maximum likelihood

The squared-shock innovation $\nu_t$ is *not* iid (heteroskedastic, skewed), so OLS /
Yule-Walker fail - GARCH is fit by MLE. Each return is conditionally Gaussian,
$r_t \mid \mathcal{F}_{t-1} \sim N(\mu, \sigma_t^2)$, and the joint density factors by
the chain rule even though returns are dependent:

$$p(r_1,\dots,r_T) = \prod_{t=1}^T p(r_t \mid \mathcal{F}_{t-1})
\;\Rightarrow\;
\ell(\theta) = -\frac12\sum_{t=1}^T\!\left[\ln(2\pi) + \ln\sigma_t^2 + \frac{\epsilon_t^2}{\sigma_t^2}\right]$$

The two data terms are in tension: $\ln\sigma_t^2$ penalises claiming large variance
everywhere; $\epsilon_t^2/\sigma_t^2$ penalises a big shock landing on a low-vol day.
MLE tunes $(\omega,\alpha,\beta)$ so $\sigma_t^2$ tracks the actual squared shocks -
literally fitting the clustering.

**Key subtlety**: $\sigma_t^2$ is not observed - it is unrolled from the recursion,
seeded at $\sigma_1^2 = $ sample variance. Each likelihood evaluation requires running
the recursion forward, so estimation is iterative numerical optimisation with the
recursion nested inside. No closed form.

**Constraints**: $\omega > 0,\ \alpha \ge 0,\ \beta \ge 0$ (positive variance),
$\alpha + \beta < 1$ (stationarity).

## From-scratch estimator (what `arch` does internally)

```python
import numpy as np
from scipy.optimize import minimize

def garch_negloglik(theta, returns):
    """Negative conditional log-likelihood of GARCH(1,1)."""
    mu, omega, alpha, beta = theta
    eps = returns - mu
    n = len(eps)

    sigma2 = np.empty(n)
    sigma2[0] = np.var(returns)              # seed with unconditional variance
    for t in range(1, n):                    # recursion: MUST loop (sequential dep.)
        sigma2[t] = omega + alpha * eps[t-1]**2 + beta * sigma2[t-1]

    ll = -0.5 * np.sum(np.log(sigma2) + eps**2 / sigma2)   # drop 2pi constant
    return -ll                               # minimise(-ll) = maximise(ll)

def fit_garch(returns):
    theta0 = [returns.mean(), 0.1*np.var(returns), 0.05, 0.90]
    bounds = [(None, None), (1e-8, None), (0.0, 1.0), (0.0, 1.0)]
    constraints = [{"type": "ineq", "fun": lambda th: 1 - th[2] - th[3]}]  # a+b<1
    return minimize(garch_negloglik, theta0, args=(returns,),
                    method="SLSQP", bounds=bounds, constraints=constraints)
```

Three things this code makes concrete:
- the **`for` loop is irreducible** - each $\sigma_t^2$ needs $\sigma_{t-1}^2$, genuine
  sequential dependence (unlike the path simulator, where independent increments let
  `cumsum` vectorise).
- **negate** the log-likelihood because `scipy` minimises.
- **SLSQP + inequality constraint** enforces the stationarity condition $\alpha+\beta<1$;
  bounds enforce positivity.

`arch` does exactly this, plus analytic gradients, better initialisation, and parameter
standard

In [3]:
from arch import arch_model

model = arch_model(returns_pct, mean="Constant", vol="GARCH", p=1, q=1, dist="normal")
res = model.fit(disp="off")
print(res.summary())

                     Constant Mean - GARCH Model Results                      
Dep. Variable:                  Close   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -3237.81
Distribution:                  Normal   AIC:                           6483.62
Method:            Maximum Likelihood   BIC:                           6506.93
                                        No. Observations:                 2511
Date:                Wed, Jun 24 2026   Df Residuals:                     2510
Time:                        18:50:52   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.0838  1.498e-02      5.592  2.238e-08 [5.442e-0

In [4]:
omega = res.params["omega"]
alpha = res.params["alpha[1]"]
beta  = res.params["beta[1]"]

persistence = alpha + beta
half_life   = np.log(0.5) / np.log(persistence)
lr_var      = omega / (1 - persistence)           # percent^2, daily
lr_vol_ann  = np.sqrt(lr_var) * np.sqrt(252) / 100  # un-scale the 100x, annualise

print(f"persistence  = {persistence:.4f}")
print(f"half-life    = {half_life:.1f} trading days")
print(f"long-run vol = {lr_vol_ann:.2%} annualised")

persistence  = 0.9708
half-life    = 23.4 trading days
long-run vol = 17.66% annualised


**GARCH 30-day forecast**

In [5]:
fc = res.forecast(horizon=21, reindex=False)
var_path = fc.variance.values[-1]          # forecast daily variances (percent^2)
avg_daily_var = var_path.mean()            # average over the 21-day option life
garch_30d_vol = np.sqrt(avg_daily_var) * np.sqrt(252) / 100   # annualised, un-scaled
print(f"GARCH 30-day-ahead annualised vol: {garch_30d_vol:.2%}")

GARCH 30-day-ahead annualised vol: 18.24%


In [7]:
from models.option_chain import (
    fetch_chain_cboe, find_closest_expiry, filter_by_expiry, clean_chain
)
from models.implied_vol import compute_smile

all_options, spot = fetch_chain_cboe("^SPX")
spxw = all_options[all_options["root"] == "SPXW"]

expiries = sorted(spxw["expiry"].unique())
expiry_30, days_30 = find_closest_expiry(expiries, target_days=30)
T_30 = days_30 / 365
print(f"30 DTE -> {expiry_30} ({days_30} days), spot {spot}")

calls_30, _ = filter_by_expiry(spxw, expiry_30)
calls_30_clean = clean_chain(calls_30, source="cboe")     # the refactored call
smile_30 = compute_smile(calls_30_clean, spot=spot, T=T_30, r=0.045, option_type="call")
smile_30

30 DTE -> 2026-07-24 (30 days), spot 7353.6299
Started: 180 rows
After bid > 0:               174
After spread filter:         163
After volume filter:         45
After open interest filter:  44
After staleness filter:      44


,option,bid,bid_size,ask,ask_size,iv,openInterest,volume,delta,gamma,...,tick,last_trade_price,lastTradeDate,percent_change,prev_day_close,root,expiry,option_type,strike,mid
20070,SPXW260724C07370000,144.00,8.0,144.90,8.0,0.165362,25.0,7.0,0.5243,0.0011,...,up,145.05,2026-06-24 13:41:28+00:00,-7.992390,157.650002,SPXW,2026-07-24,call,7370.0,144.450
20074,SPXW260724C07380000,138.00,8.0,138.90,8.0,0.164118,34.0,8.0,0.5129,0.0012,...,up,155.56,2026-06-24 13:02:52+00:00,2.747690,151.400002,SPXW,2026-07-24,call,7380.0,138.450
20076,SPXW260724C07390000,132.00,12.0,133.10,1.0,0.162859,25.0,16.0,0.5013,0.0012,...,down,142.50,2026-06-24 13:09:23+00:00,-1.893290,145.250000,SPXW,2026-07-24,call,7390.0,132.550
20078,SPXW260724C07400000,126.60,1.0,127.30,1.0,0.161819,137.0,40.0,0.4896,0.0012,...,up,127.35,2026-06-24 13:41:28+00:00,-8.512930,139.199997,SPXW,2026-07-24,call,7400.0,126.950
20080,SPXW260724C07410000,120.60,8.0,121.50,8.0,0.160283,33.0,12.0,0.4777,0.0012,...,no_change,147.63,2026-06-24 12:16:30+00:00,10.833300,133.199997,SPXW,2026-07-24,call,7410.0,121.050
20084,SPXW260724C07425000,112.50,8.0,113.30,1.0,0.158544,74.0,99.0,0.4595,0.0012,...,no_change,117.01,2026-06-24 13:22:08+00:00,-6.016060,124.500000,SPXW,2026-07-24,call,7425.0,112.900
20086,SPXW260724C07430000,109.80,10.0,110.60,1.0,0.157911,28.0,5.0,0.4534,0.0012,...,down,119.07,2026-06-24 13:13:52+00:00,-2.161050,121.700001,SPXW,2026-07-24,call,7430.0,110.200
20090,SPXW260724C07450000,99.40,1.0,100.20,10.0,0.155483,778.0,71.0,0.4286,0.0012,...,down,118.00,2026-06-24 12:36:46+00:00,6.594400,110.700001,SPXW,2026-07-24,call,7450.0,99.800
20096,SPXW260724C07475000,87.00,13.0,87.90,9.0,0.152359,242.0,5.0,0.3971,0.0012,...,up,110.39,2026-06-24 12:22:45+00:00,13.162500,97.549999,SPXW,2026-07-24,call,7475.0,87.450
20100,SPXW260724C07490000,80.10,10.0,80.80,1.0,0.150500,51.0,8.0,0.3779,0.0012,...,down,86.10,2026-06-24 13:16:28+00:00,-4.545460,90.200001,SPXW,2026-07-24,call,7490.0,80.450


In [8]:
atm_idx = (smile_30["strike"] - spot).abs().idxmin()
print(f"ATM strike {smile_30.loc[atm_idx, 'strike']}, implied vol {smile_30.loc[atm_idx, 'iv']:.2%}")

ATM strike 7370.0, implied vol 16.54%


# Lessons: Implied vs Realised Volatility, and What the Gap Tells You

## What we did

Two independent estimates of S&P 500 volatility, from two different worlds:

1. **Implied vol** - inverted from option *prices* (Brent root-find, Week 1). The
   market's forward-looking, risk-neutral expectation of vol over the option's life.
   Read off the 30 DTE ATM strike: **16.5%** annualised.
2. **Realised vol** - forecast from a GARCH(1,1) fit to historical *returns* (this
   notebook). Backward-looking, physical-measure, model-based. 30-day-ahead forecast:
   **18.2%** annualised.

Both on the same day (Jun 24), both over the same 30-day horizon - so the comparison
is fair. Implied sits ~1.7 vol points *below* the realised forecast.

## Why matched horizons matter (the key methodological point)

The naive comparison - GARCH *long-run* vol (17.7%, a decade average including COVID)
vs a single implied number - is apples-to-oranges. GARCH's long-run $\bar\sigma$, its
*current conditional* $\sigma_t$, and a *30-day implied* are three different horizons.
Because GARCH mean-reverts geometrically, the right realised number to compare against
30 DTE implied is the **variance-path-averaged 30-day forecast**, not the long-run mean
and not today's spot vol. Matching horizon is what turns a vague "vol is ~17%" into a
defensible 18.2% vs 16.5%.

## The two volatilities measure genuinely different things

| | Implied | Realised (GARCH) |
|---|---|---|
| Source | option prices | return history |
| Direction | forward-looking | backward-looking |
| Measure | risk-neutral $\mathbb{Q}$ | physical $\mathbb{P}$ |
| Interpretation | what the market *charges* for vol | what the model *expects* vol to be |

The measure difference is not a technicality - it is the **variance risk premium**.
Options are insurance; sellers demand compensation for bearing vol risk, so implied
*usually* sits *above* realised (you pay a premium for protection). Here implied is
*below* realised, the rarer configuration.

## How a trader reads the gap (intuition)

The core relationship a delta-hedged option position lives or dies by is the
Gamma-Theta identity (Week-1 Greeks reference):

$$\Theta \approx -\tfrac12 \sigma_{\text{impl}}^2 S^2 \Gamma \quad\text{(delta-hedged)}$$

You *buy* an option at the **implied** vol (that is the price). You then delta-hedge and
collect gamma P&L as the underlying actually moves - and the underlying moves at the
**realised** vol. Over the life of the hedge, the P&L of a delta-hedged long option is
approximately:

$$\text{P\&L} \;\propto\; \tfrac12\,S^2\,\Gamma\,\big(\sigma_{\text{real}}^2 - \sigma_{\text{impl}}^2\big)\,\Delta t$$

So the entire game is the sign of $\sigma_{\text{real}} - \sigma_{\text{impl}}$:

- **Realised > Implied** (our case: 18.2 > 16.5): the underlying moves *more* than the
  option price assumed. Gamma profit from rebalancing exceeds the theta you bleed.
  -> options look **cheap**; the signal is **long vol** (buy straddle, delta-hedge).
- **Realised < Implied**: the underlying is calmer than the option charged for. Theta
  bleed exceeds gamma profit. -> options look **rich**; the signal is **short vol**
  (sell straddle, delta-hedge - collect the premium).

This is one of the four uses of IV from Week 1 (the **implied-vs-realised arbitrage**),
now concrete: a 1.7-point edge in favour of being long 30-day vol *if* GARCH's forecast
is trusted.

## What we can actually extract from this data

- A **fair-value vol estimate** to sanity-check market prices against - "is the
  option chain pricing vol high or low relative to recent behaviour?"
- A **directional vol signal** (long/short vol) with a sign and a magnitude, feeding a
  delta-hedged straddle strategy - which is exactly the **Phase 3 / Week 14** deliverable.
- A **vol term structure** to compare against the implied term structure (the 30 vs 75
  DTE smiles from Week 1): GARCH *predicts* a term structure from the persistence
  $\alpha+\beta$ (vol mean-reverts, so longer-horizon forecasts pull toward $\bar\sigma$).
  Not done here - a natural extension.

## Caveats (a fair comparison names its weaknesses)

- **No dividend in the IV inversion**: SPX yields ~1.3%, lowering the ATM forward below
  spot and nudging implied. The carry term $b = r - q$ (Day 9) is the fix; effect is
  small (~0.1-0.2 pt) but real.
- **Gaussian GARCH**: equity returns are fat-tailed; `dist="t"` may shift the forecast.
- **Point estimate**: the 18.2% ignores parameter uncertainty (std errors on
  $\alpha,\beta$) and model risk. The gap is a signal, not a certainty.
- **One snapshot**: a single day's gap is noise; a real strategy needs the gap measured
  over time, with the variance risk premium accounted for (implied being *structurally*
  a bit above realised on average is normal, not necessarily an edge).